# Chapter 6 — Multi-Head Self-Attention

> Course: **llm.c — Zero to Hero**, Chapter 6 of ~20.
> Builds on Chapters 3–4 (LayerNorm reductions, matmul backward, two-pass parallel patterns).

This is the **signature operation of the Transformer**, the longest single function in `train_gpt2.c` (~75 lines for `attention_forward + attention_backward`), and the layer that makes language models *language* models.

You already know what attention is from `nn.MultiheadAttention` — but you may not have seen it written without abstractions. In `llm.c` it is **just four nested loops** and an explicit causal mask, and once you can read it, you understand attention better than 90% of people who use it.

This chapter is going to be longer than usual. Plan for it.

### Learning objectives

By the end of this chapter you will:

- Know the **`(B, T, 3C)` packed QKV layout** that `llm.c` (and most production Transformers) use.
- Read and write `attention_forward` — the **four-pass** structure (QK·scale → max → exp+sum → softmax+mask → V accumulate).
- Explain the **multi-head split** as nothing more than a clever reshape — the math is identical to single-head.
- Read and write `attention_backward` — three sub-backwards (V accumulation, softmax Jacobian, QK matmul) that you can derive from the chain rule on a piece of paper.
- Cross-check the whole thing numerically against PyTorch's `F.scaled_dot_product_attention`.


## 1. Concept — Self-Attention as a "Soft Dictionary"

You know the math:

$$\text{Attn}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$

The intuition that's worth holding on to as we write C: **for each position `t`, attention asks every earlier position "how much do you match my query?", normalizes those scores into a probability distribution, then averages those positions' values weighted by that distribution.**

In code that's four steps **per `(batch, position, head)` tuple**:

1. **QK·scale** — dot products of one Q vector against all K vectors → `T` scores
2. **softmax** — turn scores into weights that sum to 1, **masking out future positions**
3. **Accumulate** — weighted sum of V vectors using those weights → one output vector

(Pass 2 inside `llm.c` is split into two passes — once to compute the max for stability, once to compute exp+sum — but conceptually it's one softmax.)

The thing that makes attention different from every other Transformer layer: **it's the only one that mixes information across time.** LayerNorm, GELU, residuals, matmul — all of those run independently at each `(b, t)`. Attention is where the model gets to look at history.


## 2. Concept — The `(B, T, 3C)` Packed QKV Layout

Most Transformer implementations compute Q, K, V from a single linear projection of the previous layer's output:

```
qkv = nn.Linear(C, 3*C)(layernorm_output)   # shape (B, T, 3C)
q, k, v = qkv.chunk(3, dim=-1)
```

`llm.c` keeps the result of that projection **packed**: a single contiguous `(B, T, 3*C)` tensor where:

- `inp[b, t,    0 :   C]` = Q at position `(b, t)`
- `inp[b, t,    C : 2*C]` = K at position `(b, t)`
- `inp[b, t, 2*C : 3*C]` = V at position `(b, t)`

So when the C code wants the key at position `(b, t2)` for head `h` it computes:

```c
float* key_t2 = inp + b*T*C3 + t2*C3 + h*hs + C;   // C3 = 3*C, hs = C/NH
//              ^^^ batch    ^^^ time   ^^^ head    ^^^ +C to skip Q
```

Read it left to right: skip past `b` batches' worth of `T*C3` floats, then `t2` timesteps, then `h` heads' worth of `hs` floats *within the Q region*, then `+C` to land in the K region. **The whole layout is a stride trick.** No data is ever copied.


## 3. Concept — Multi-Head is Just Slicing

Multi-head attention splits the channel dimension `C` into `NH` heads of size `hs = C / NH`. **Each head is an independent attention computation** on its own slice of Q/K/V — heads don't talk to each other inside attention. The outputs are concatenated back into `C` channels.

Crucially: there is **no extra parameter or new math** for multi-head. The QKV linear is *the same projection*; the math just operates on `hs`-dim sub-vectors instead of full-`C` vectors. The benefit is that different heads end up specializing on different patterns (one head learns "verb-subject", another learns "matching brackets", etc.).

For us the only consequence is one more `for (h)` loop in the code — and one more head index in every pointer computation.


## 4. PyTorch Baseline (Hand-Rolled)

We won't use `nn.MultiheadAttention` because its API hides what we want to see. Let's write attention by hand using basic tensor ops — much closer to what the C does.


In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)
B, T, C, NH = 2, 4, 8, 2     # 2 heads, head size hs = 4
hs = C // NH

# Build random Q, K, V and pack them as (B, T, 3C) — same layout as llm.c's `inp`
q = torch.randn(B, T, C)
k = torch.randn(B, T, C)
v = torch.randn(B, T, C)
qkv = torch.cat([q, k, v], dim=-1)   # (B, T, 3*C)
print("qkv shape:", qkv.shape)

# Reshape into per-head views: (B, NH, T, hs)
def split_heads(x): return x.view(B, T, NH, hs).transpose(1, 2)
qh, kh, vh = split_heads(q), split_heads(k), split_heads(v)

# Scaled dot-product with causal mask, by hand
scale  = hs ** -0.5
scores = (qh @ kh.transpose(-2, -1)) * scale          # (B, NH, T, T)
mask   = torch.tril(torch.ones(T, T, dtype=torch.bool))
scores = scores.masked_fill(~mask, float('-inf'))
att    = F.softmax(scores, dim=-1)
out    = att @ vh                                     # (B, NH, T, hs)
out    = out.transpose(1, 2).contiguous().view(B, T, C)
print("output shape:", out.shape)
print("output[0, 0, :4]:", out[0, 0, :4].tolist())


Three knobs to remember from this baseline:

- **Causal mask**: `torch.tril(...)` gives a `(T, T)` lower-triangular matrix. We replace the upper triangle of `scores` with `-inf`, so `softmax` puts zero probability on future tokens.
- **Scale by `1/√hs`**: scales scores so their variance is roughly 1 regardless of head size — keeps softmax from saturating for large `hs`.
- **Per-head transpose / view**: the `(B, T, NH, hs).transpose(1, 2) → (B, NH, T, hs)` is *just a stride trick* in PyTorch (no data copy). The C code achieves the same effect by computing pointers with `h*hs` offsets.


## 5. The C Forward — Four Passes Per `(b, t, h)`

From [`train_gpt2.c`](train_gpt2.c) lines 271–345. Here's the full function with the OpenMP pragma intact:

```c
void attention_forward(float* out, float* preatt, float* att,
                       float* inp,
                       int B, int T, int C, int NH) {
    // input is (B, T, 3C) holding the query, key, value (Q, K, V) vectors
    // preatt, att are (B, NH, T, T)
    // output is (B, T, C)
    int C3 = C*3;
    int hs = C / NH;
    float scale = 1.0 / sqrtf(hs);

    #pragma omp parallel for collapse(3)
    for (int b = 0; b < B; b++) {
        for (int t = 0; t < T; t++) {
            for (int h = 0; h < NH; h++) {
                float* query_t    = inp    + b*T*C3 + t*C3 + h*hs;
                float* preatt_bth = preatt + b*NH*T*T + h*T*T + t*T;
                float* att_bth    = att    + b*NH*T*T + h*T*T + t*T;

                // PASS 1: query·key (scaled), track maxval
                float maxval = -10000.0f;
                for (int t2 = 0; t2 <= t; t2++) {                      // <-- causal: only t2 <= t
                    float* key_t2 = inp + b*T*C3 + t2*C3 + h*hs + C;   // +C: skip Q to land in K
                    float val = 0.0f;
                    for (int i = 0; i < hs; i++) val += query_t[i] * key_t2[i];
                    val *= scale;
                    if (val > maxval) maxval = val;
                    preatt_bth[t2] = val;
                }

                // PASS 2: subtract max for stability, then exp, accumulate sum
                float expsum = 0.0f;
                for (int t2 = 0; t2 <= t; t2++) {
                    float expv = expf(preatt_bth[t2] - maxval);
                    expsum += expv;
                    att_bth[t2] = expv;
                }
                float expsum_inv = expsum == 0.0f ? 0.0f : 1.0f / expsum;

                // PASS 3: normalize to softmax; explicitly zero the masked positions
                for (int t2 = 0; t2 < T; t2++) {
                    if (t2 <= t) att_bth[t2] *= expsum_inv;
                    else         att_bth[t2] = 0.0f;
                }

                // PASS 4: weighted sum of value vectors -> out[b, t, h*hs : h*hs+hs]
                float* out_bth = out + b*T*C + t*C + h*hs;
                for (int i = 0; i < hs; i++) out_bth[i] = 0.0f;
                for (int t2 = 0; t2 <= t; t2++) {
                    float* value_t2 = inp + b*T*C3 + t2*C3 + h*hs + C*2;  // +2C: skip Q,K to V
                    float att_btht2 = att_bth[t2];
                    for (int i = 0; i < hs; i++) out_bth[i] += att_btht2 * value_t2[i];
                }
            }
        }
    }
}
```

### Pass-by-pass annotation

| Pass | What | Loop body |
|---|---|---|
| 1 | dot products + scale + find max | `preatt_bth[t2] = scale·Q·K[t2]`, track `maxval` |
| 2 | shift-then-exp for stability | `att_bth[t2] = exp(preatt_bth[t2] - maxval)`, accumulate `expsum` |
| 3 | normalize and mask | `att_bth[t2] *= 1/expsum` for `t2 <= t`, else `0` |
| 4 | weighted sum of values | `out_bth += att_bth[t2] * V[t2]` |

### Three things worth pausing on

1. **The causal mask is implemented by `t2 <= t`**, not by setting `-inf` in the score matrix. Inside passes 1, 2, 4 we simply *skip* future positions. Pass 3 then explicitly stores `0` in the masked entries so the saved `att` tensor matches what PyTorch would produce — useful for debugging, not strictly required.
2. **`maxval = -10000.0f`** is a hack — for very negative pre-attention scores the code may lose accuracy. The CUDA version uses `-FLT_MAX`. The CPU version stays this way because real values rarely approach `-10000` after the `1/√hs` scaling.
3. **`#pragma omp parallel for collapse(3)`** — three independent dimensions (`b`, `t`, `h`) get flattened into `B*T*NH` parallel iterations. Every iteration writes to its own slice of `out`, `preatt`, `att`, so no race. Note that `t` *is* parallelized — the **causal dependence is intra-`t2`, not intra-`t`**. Each `t` independently does its dot products against all earlier `t2 ≤ t`.


## 6. Translation Bridge

| PyTorch | C in `llm.c` |
|---|---|
| `Q, K, V = qkv.chunk(3, dim=-1)` | Pointers `inp + h*hs`, `inp + h*hs + C`, `inp + h*hs + 2C` — no copy |
| `q.view(B,T,NH,hs).transpose(1,2)` | The `+ h*hs` offset inside the Q region |
| `scores = q @ k.T * scale` | `for (i) val += query_t[i] * key_t2[i]; val *= scale;` |
| `scores.masked_fill(~tril, -inf)` | `for (t2 = 0; t2 <= t; t2++)` — never touches future positions |
| `F.softmax(scores, dim=-1)` | The two-pass max-subtract / exp-sum / normalize block |
| `att @ v` | `out_bth[i] += att_bth[t2] * value_t2[i]` |
| `out.transpose(1,2).reshape(B,T,C)` | The output is *already* `(B,T,C)` because `out_bth` writes into the `h*hs` slice |

Mental model: **multi-head attention in C is the PyTorch implementation with all the views and reshapes inlined as pointer offsets.** Once you can recompute `inp + b*T*C3 + t2*C3 + h*hs + C` from "key of head h at position t2 in batch b", you've got it.


## 7. Compile and Verify Forward Against PyTorch

In [ ]:
!mkdir -p course/ch06_build


In [ ]:
%%writefile course/ch06_build/attention_forward.c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <omp.h>

void attention_forward(float* out, float* preatt, float* att,
                       float* inp,
                       int B, int T, int C, int NH) {
    int C3 = C*3;
    int hs = C / NH;
    float scale = 1.0f / sqrtf((float)hs);

    #pragma omp parallel for collapse(3)
    for (int b = 0; b < B; b++)
        for (int t = 0; t < T; t++)
            for (int h = 0; h < NH; h++) {
                float* query_t    = inp    + b*T*C3 + t*C3 + h*hs;
                float* preatt_bth = preatt + b*NH*T*T + h*T*T + t*T;
                float* att_bth    = att    + b*NH*T*T + h*T*T + t*T;

                float maxval = -10000.0f;
                for (int t2 = 0; t2 <= t; t2++) {
                    float* key_t2 = inp + b*T*C3 + t2*C3 + h*hs + C;
                    float val = 0.0f;
                    for (int i = 0; i < hs; i++) val += query_t[i] * key_t2[i];
                    val *= scale;
                    if (val > maxval) maxval = val;
                    preatt_bth[t2] = val;
                }

                float expsum = 0.0f;
                for (int t2 = 0; t2 <= t; t2++) {
                    float expv = expf(preatt_bth[t2] - maxval);
                    expsum += expv;
                    att_bth[t2] = expv;
                }
                float expsum_inv = expsum == 0.0f ? 0.0f : 1.0f / expsum;

                for (int t2 = 0; t2 < T; t2++) {
                    if (t2 <= t) att_bth[t2] *= expsum_inv;
                    else         att_bth[t2] = 0.0f;
                }

                float* out_bth = out + b*T*C + t*C + h*hs;
                for (int i = 0; i < hs; i++) out_bth[i] = 0.0f;
                for (int t2 = 0; t2 <= t; t2++) {
                    float* value_t2 = inp + b*T*C3 + t2*C3 + h*hs + C*2;
                    float a = att_bth[t2];
                    for (int i = 0; i < hs; i++) out_bth[i] += a * value_t2[i];
                }
            }
}

static void* rd(const char* p, size_t n) {
    FILE* f = fopen(p, "rb"); if (!f){perror(p); exit(1);}
    void* b = malloc(n); size_t r = fread(b,1,n,f); (void)r; fclose(f); return b;
}

int main(int argc, char** argv) {
    if (argc != 5) { fprintf(stderr, "usage: B T C NH\n"); return 1; }
    int B=atoi(argv[1]), T=atoi(argv[2]), C=atoi(argv[3]), NH=atoi(argv[4]);
    float* inp    = (float*) rd("course/ch06_build/inp.bin", (size_t)B*T*3*C*sizeof(float));
    float* out    = (float*) malloc((size_t)B*T*C*sizeof(float));
    float* preatt = (float*) malloc((size_t)B*NH*T*T*sizeof(float));
    float* att    = (float*) malloc((size_t)B*NH*T*T*sizeof(float));
    attention_forward(out, preatt, att, inp, B, T, C, NH);
    FILE* f;
    f=fopen("course/ch06_build/out.bin",   "wb"); fwrite(out,   4,(size_t)B*T*C,f);   fclose(f);
    f=fopen("course/ch06_build/preatt.bin","wb"); fwrite(preatt,4,(size_t)B*NH*T*T,f);fclose(f);
    f=fopen("course/ch06_build/att.bin",   "wb"); fwrite(att,   4,(size_t)B*NH*T*T,f);fclose(f);
    free(inp); free(out); free(preatt); free(att);
    return 0;
}


In [ ]:
!gcc -O3 -Wall -fopenmp -o course/ch06_build/attention_forward course/ch06_build/attention_forward.c -lm


In [ ]:
import numpy as np, torch, torch.nn.functional as F, subprocess
torch.manual_seed(0)
B, T, C, NH = 2, 4, 8, 2
hs = C // NH

q = torch.randn(B, T, C); k = torch.randn(B, T, C); v = torch.randn(B, T, C)
qkv = torch.cat([q, k, v], dim=-1)               # (B, T, 3C) — same layout as llm.c
qkv.numpy().astype(np.float32).tofile("course/ch06_build/inp.bin")

subprocess.run(["./course/ch06_build/attention_forward", str(B), str(T), str(C), str(NH)], check=True)
out_c = np.fromfile("course/ch06_build/out.bin", dtype=np.float32).reshape(B, T, C)
att_c = np.fromfile("course/ch06_build/att.bin", dtype=np.float32).reshape(B, NH, T, T)

# PyTorch reference (hand-rolled, matching llm.c semantics exactly)
def heads(x): return x.view(B, T, NH, hs).transpose(1, 2)
qh, kh, vh = heads(q), heads(k), heads(v)
scores = (qh @ kh.transpose(-2, -1)) * (hs ** -0.5)
mask = torch.tril(torch.ones(T, T, dtype=torch.bool))
scores = scores.masked_fill(~mask, float('-inf'))
att_pt = F.softmax(scores, dim=-1)               # (B, NH, T, T)
# llm.c sets att = 0 (not the softmax-of-(-inf)) at masked positions; match that:
att_pt = att_pt.masked_fill(~mask, 0.0)
out_pt = (att_pt @ vh).transpose(1, 2).contiguous().view(B, T, C)

print(f"out diff:   {np.max(np.abs(out_c - out_pt.numpy())):.2e}")
print(f"att diff:   {np.max(np.abs(att_c - att_pt.numpy())):.2e}")


## 8. Toy Trace — One Head, One Position

To make the four passes feel concrete, let's hand-trace a single `(b=0, t=2, h=0)` triple with `T=4, hs=2` and pre-cooked Q/K/V values.


In [ ]:
%%writefile course/ch06_build/toy_attention.c
#include <stdio.h>
#include <math.h>

int main(void) {
    const int hs = 2;
    // We're at position t=2, so we look at t2 in {0, 1, 2}.
    float Q[2]    = {1.0f, 0.0f};                 // query at t=2
    float Ks[3*2] = {1,0,  0,1,  1,1};            // keys at t2=0,1,2
    float Vs[3*2] = {10,20, 30,40, 50,60};        // values at t2=0,1,2

    float scale = 1.0f / sqrtf((float)hs);
    printf("scale = 1/sqrt(%d) = %.4f\n", hs, scale);

    // Pass 1: dot products + max
    float scores[3]; float maxval = -1e30f;
    for (int t2 = 0; t2 < 3; t2++) {
        float dot = 0; for (int i = 0; i < hs; i++) dot += Q[i] * Ks[t2*hs+i];
        scores[t2] = dot * scale;
        if (scores[t2] > maxval) maxval = scores[t2];
        printf("  t2=%d  Q.K = %.3f -> score = %.4f\n", t2, scores[t2]/scale, scores[t2]);
    }
    printf("maxval = %.4f\n", maxval);

    // Pass 2: exp, sum
    float att[3]; float expsum = 0;
    for (int t2 = 0; t2 < 3; t2++) { att[t2] = expf(scores[t2]-maxval); expsum += att[t2]; }
    printf("expsum = %.4f\n", expsum);

    // Pass 3: normalize
    for (int t2 = 0; t2 < 3; t2++) att[t2] /= expsum;
    printf("att = [%.4f %.4f %.4f]   (sums to %.4f)\n", att[0], att[1], att[2], att[0]+att[1]+att[2]);

    // Pass 4: weighted sum of values
    float out[2] = {0,0};
    for (int t2 = 0; t2 < 3; t2++)
        for (int i = 0; i < hs; i++) out[i] += att[t2] * Vs[t2*hs+i];
    printf("out = [%.4f %.4f]   (a convex combination of [10,20], [30,40], [50,60])\n", out[0], out[1]);
    return 0;
}


In [ ]:
!gcc -O2 -Wall -o course/ch06_build/toy_attention course/ch06_build/toy_attention.c -lm && ./course/ch06_build/toy_attention


Notice that `out` is **a convex combination** of the V vectors — that's a property of softmax weights summing to 1. Attention literally cannot do anything other than mix existing values via a probability distribution. That's why people call attention's output a *soft lookup* into the V table.


## 9. The Backward — Three Sub-Backwards

[`train_gpt2.c`](train_gpt2.c) lines 347–405. The function structure is:

```
                    forward    →    backward (3 stages, in reverse)
PASS 4: out = att·V              →  Stage A: dout → datt, dV
PASS 2&3: att = softmax(preatt)  →  Stage B: datt → dpreatt   (softmax Jacobian)
PASS 1:  preatt = scale·Q·K      →  Stage C: dpreatt → dQ, dK
```

### Stage A — through the value accumulation

For `out_bth[i] = Σ_{t2 ≤ t} att[t2] * V_t2[i]`, the partials are:

$$\frac{\partial L}{\partial \text{att}[t_2]} = \sum_i V_{t_2}[i] \cdot \text{dout}[i]$$
$$\frac{\partial L}{\partial V_{t_2}[i]} = \text{att}[t_2] \cdot \text{dout}[i]$$

Both are written as `+=` because the same buffers receive contributions from multiple `(b, t, h)`.

### Stage B — through softmax

The softmax `att = softmax(preatt)` has Jacobian:

$$\frac{\partial \text{att}[t_2]}{\partial \text{preatt}[t_3]} = \text{att}[t_2]\,(\delta_{t_2, t_3} - \text{att}[t_3])$$

So `dpreatt[t3] = Σ_{t2} datt[t2] * att[t2] * (δ_{t2,t3} - att[t3])`. The C code writes this as the explicit `T × T` Jacobian-vector product — easy to read, suboptimal in FLOPs (the fused form `att[t3] * (datt[t3] - Σ datt·att)` halves the work, and we'll see it on the GPU side in Chapter 12).

Notice that the softmax backward **does not need the input `preatt`** — only the output `att` (and `datt`). That's a general property of softmax, like ReLU and tanh. `llm.c` therefore caches `att`, not `preatt`, for backward.

### Stage C — through the QK matmul

For `preatt[t2] = scale · Σ_i Q[i] · K_t2[i]`:

$$\frac{\partial L}{\partial Q[i]} \mathrel{+}= \sum_{t_2 \le t} \text{scale} \cdot K_{t_2}[i] \cdot \text{dpreatt}[t_2]$$
$$\frac{\partial L}{\partial K_{t_2}[i]} \mathrel{+}= \text{scale} \cdot Q[i] \cdot \text{dpreatt}[t_2]$$

Both `dQ` and `dK` get written into different slices of the same big `dinp` buffer (`+0` for Q, `+C` for K, `+2C` for V) — same packed layout as the forward.


### The full C backward

```c
void attention_backward(float* dinp, float* dpreatt, float* datt,
                        float* dout, float* inp, float* att,
                        int B, int T, int C, int NH) {
    int C3 = C*3;
    int hs = C / NH;
    float scale = 1.f / sqrtf(hs);

    for (int b = 0; b < B; b++)
        for (int t = 0; t < T; t++)
            for (int h = 0; h < NH; h++) {
                float* att_bth     = att     + b*NH*T*T + h*T*T + t*T;
                float* datt_bth    = datt    + b*NH*T*T + h*T*T + t*T;
                float* dpreatt_bth = dpreatt + b*NH*T*T + h*T*T + t*T;
                float* dquery_t    = dinp + b*T*C3 + t*C3 + h*hs;
                float* query_t     = inp  + b*T*C3 + t*C3 + h*hs;
                float* dout_bth    = dout + b*T*C  + t*C  + h*hs;

                // Stage A: through pass 4 (V accumulation)
                for (int t2 = 0; t2 <= t; t2++) {
                    float* value_t2  = inp  + b*T*C3 + t2*C3 + h*hs + C*2;
                    float* dvalue_t2 = dinp + b*T*C3 + t2*C3 + h*hs + C*2;
                    for (int i = 0; i < hs; i++) {
                        datt_bth[t2]   += value_t2[i] * dout_bth[i];
                        dvalue_t2[i]   += att_bth[t2] * dout_bth[i];
                    }
                }

                // Stage B: through pass 2&3 (softmax Jacobian, T x T)
                for (int t2 = 0; t2 <= t; t2++)
                    for (int t3 = 0; t3 <= t; t3++) {
                        float indicator = (t2 == t3) ? 1.0f : 0.0f;
                        float local_derivative = att_bth[t2] * (indicator - att_bth[t3]);
                        dpreatt_bth[t3] += local_derivative * datt_bth[t2];
                    }

                // Stage C: through pass 1 (Q·K matmul)
                for (int t2 = 0; t2 <= t; t2++) {
                    float* key_t2  = inp  + b*T*C3 + t2*C3 + h*hs + C;
                    float* dkey_t2 = dinp + b*T*C3 + t2*C3 + h*hs + C;
                    for (int i = 0; i < hs; i++) {
                        dquery_t[i]    += key_t2[i] * dpreatt_bth[t2] * scale;
                        dkey_t2[i]     += query_t[i] * dpreatt_bth[t2] * scale;
                    }
                }
            }
}
```

### Why is there no `#pragma omp` on this backward?

This is the only major op in `train_gpt2.c` whose backward is **not parallelized**. Look at Stage A and Stage C: each `(b, t, h)` writes into `dvalue_t2`, `dkey_t2` indexed by `t2` — and `t2` ranges over **other** time steps. Two threads working on different `t` values can both write to `dV[b, t2=3, h]`, racing on the same memory. So a `parallel for collapse(3)` would be incorrect.

A correct parallel backward would need either per-thread accumulators (more memory) or atomics (slow). The CPU code stays serial; the CUDA version uses different parallelism altogether (we'll see this in Chapter 16).


## 10. Compile and Verify Backward

In [ ]:
%%writefile course/ch06_build/attention_backward.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>

void attention_backward(float* dinp, float* dpreatt, float* datt,
                        float* dout, float* inp, float* att,
                        int B, int T, int C, int NH) {
    int C3 = C*3;
    int hs = C / NH;
    float scale = 1.0f / sqrtf((float)hs);

    for (int b = 0; b < B; b++)
        for (int t = 0; t < T; t++)
            for (int h = 0; h < NH; h++) {
                float* att_bth     = att     + b*NH*T*T + h*T*T + t*T;
                float* datt_bth    = datt    + b*NH*T*T + h*T*T + t*T;
                float* dpreatt_bth = dpreatt + b*NH*T*T + h*T*T + t*T;
                float* dquery_t    = dinp + b*T*C3 + t*C3 + h*hs;
                float* query_t     = inp  + b*T*C3 + t*C3 + h*hs;
                float* dout_bth    = dout + b*T*C  + t*C  + h*hs;

                for (int t2 = 0; t2 <= t; t2++) {
                    float* value_t2  = inp  + b*T*C3 + t2*C3 + h*hs + C*2;
                    float* dvalue_t2 = dinp + b*T*C3 + t2*C3 + h*hs + C*2;
                    for (int i = 0; i < hs; i++) {
                        datt_bth[t2]   += value_t2[i] * dout_bth[i];
                        dvalue_t2[i]   += att_bth[t2] * dout_bth[i];
                    }
                }
                for (int t2 = 0; t2 <= t; t2++)
                    for (int t3 = 0; t3 <= t; t3++) {
                        float indicator = (t2 == t3) ? 1.0f : 0.0f;
                        float local = att_bth[t2] * (indicator - att_bth[t3]);
                        dpreatt_bth[t3] += local * datt_bth[t2];
                    }
                for (int t2 = 0; t2 <= t; t2++) {
                    float* key_t2  = inp  + b*T*C3 + t2*C3 + h*hs + C;
                    float* dkey_t2 = dinp + b*T*C3 + t2*C3 + h*hs + C;
                    for (int i = 0; i < hs; i++) {
                        dquery_t[i]  += key_t2[i] * dpreatt_bth[t2] * scale;
                        dkey_t2[i]   += query_t[i] * dpreatt_bth[t2] * scale;
                    }
                }
            }
}

static void* rd(const char* p, size_t n) {
    FILE* f = fopen(p, "rb"); if (!f){perror(p); exit(1);}
    void* b = malloc(n); size_t r = fread(b,1,n,f); (void)r; fclose(f); return b;
}

int main(int argc, char** argv) {
    if (argc != 5) return 1;
    int B=atoi(argv[1]), T=atoi(argv[2]), C=atoi(argv[3]), NH=atoi(argv[4]);
    float* inp  = (float*) rd("course/ch06_build/inp.bin",  (size_t)B*T*3*C*sizeof(float));
    float* att  = (float*) rd("course/ch06_build/att.bin",  (size_t)B*NH*T*T*sizeof(float));
    float* dout = (float*) rd("course/ch06_build/dout.bin", (size_t)B*T*C*sizeof(float));
    float* dinp    = (float*) calloc((size_t)B*T*3*C,  sizeof(float));
    float* dpreatt = (float*) calloc((size_t)B*NH*T*T, sizeof(float));
    float* datt    = (float*) calloc((size_t)B*NH*T*T, sizeof(float));
    attention_backward(dinp, dpreatt, datt, dout, inp, att, B, T, C, NH);
    FILE* f = fopen("course/ch06_build/dinp.bin","wb"); fwrite(dinp,4,(size_t)B*T*3*C,f); fclose(f);
    free(inp); free(att); free(dout); free(dinp); free(dpreatt); free(datt);
    return 0;
}


In [ ]:
!gcc -O3 -Wall -o course/ch06_build/attention_backward course/ch06_build/attention_backward.c -lm


In [ ]:
import numpy as np, torch, torch.nn.functional as F, subprocess
torch.manual_seed(0)
B, T, C, NH = 2, 4, 8, 2
hs = C // NH

# Build leaf q,k,v with grad enabled
q = torch.randn(B, T, C, requires_grad=True)
k = torch.randn(B, T, C, requires_grad=True)
v = torch.randn(B, T, C, requires_grad=True)

# Forward (matching llm.c semantics)
def heads(x): return x.view(B, T, NH, hs).transpose(1, 2)
qh, kh, vh = heads(q), heads(k), heads(v)
scores = (qh @ kh.transpose(-2, -1)) * (hs ** -0.5)
mask = torch.tril(torch.ones(T, T, dtype=torch.bool))
scores = scores.masked_fill(~mask, float('-inf'))
att = F.softmax(scores, dim=-1)
out = (att @ vh).transpose(1, 2).contiguous().view(B, T, C)

dout = torch.randn_like(out)
out.backward(dout)

# Save inputs the C backward needs: inp (the packed Q,K,V), att (cached from forward), dout
qkv = torch.cat([q.detach(), k.detach(), v.detach()], dim=-1)
qkv.numpy().astype(np.float32).tofile("course/ch06_build/inp.bin")
# Match the llm.c convention: zero out masked positions of att before saving
att_c_format = att.detach().masked_fill(~mask, 0.0)
att_c_format.numpy().astype(np.float32).tofile("course/ch06_build/att.bin")
dout.numpy().astype(np.float32).tofile("course/ch06_build/dout.bin")

subprocess.run(["./course/ch06_build/attention_backward",
                str(B), str(T), str(C), str(NH)], check=True)

dinp_c = np.fromfile("course/ch06_build/dinp.bin", dtype=np.float32).reshape(B, T, 3*C)
# Split the packed dinp into dQ, dK, dV
dq_c = dinp_c[:, :, 0:C]
dk_c = dinp_c[:, :, C:2*C]
dv_c = dinp_c[:, :, 2*C:3*C]

print(f"dQ diff: {np.max(np.abs(dq_c - q.grad.numpy())):.2e}")
print(f"dK diff: {np.max(np.abs(dk_c - k.grad.numpy())):.2e}")
print(f"dV diff: {np.max(np.abs(dv_c - v.grad.numpy())):.2e}")


## 11. TODO Exercise 1 — Pass 1 (Scaled Q·K with Causal Mask)

Boilerplate provided. Fill in the **dot product accumulation, the scaling, and the maxval tracking**.


In [ ]:
%%writefile course/ch06_build/exercise1.c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <omp.h>

void attention_pass1(float* preatt, float* inp, int B, int T, int C, int NH) {
    int C3 = C*3;
    int hs = C / NH;
    float scale = 1.0f / sqrtf((float)hs);

    #pragma omp parallel for collapse(3)
    for (int b = 0; b < B; b++)
        for (int t = 0; t < T; t++)
            for (int h = 0; h < NH; h++) {
                float* query_t    = inp    + b*T*C3 + t*C3 + h*hs;
                float* preatt_bth = preatt + b*NH*T*T + h*T*T + t*T;

                float maxval = -10000.0f;
                for (int t2 = 0; t2 <= t; t2++) {
                    float* key_t2 = inp + b*T*C3 + t2*C3 + h*hs + C;

                    // TODO 1: compute val = dot(query_t, key_t2) over hs elements
                    float val = 0.0f;
                    // your code here

                    // TODO 2: scale by 1/sqrt(hs)  — already in `scale`

                    // TODO 3: update maxval if val > maxval

                    preatt_bth[t2] = val;
                }
                // (Causal positions t2 > t are intentionally left as-is — they don't matter.)
                (void)maxval;
            }
}

static void* rd(const char* p, size_t n){FILE*f=fopen(p,"rb");void*b=malloc(n);size_t r=fread(b,1,n,f);(void)r;fclose(f);return b;}

int main(int argc, char** argv) {
    int B=atoi(argv[1]), T=atoi(argv[2]), C=atoi(argv[3]), NH=atoi(argv[4]);
    float* inp    = (float*) rd("course/ch06_build/inp.bin", (size_t)B*T*3*C*sizeof(float));
    float* preatt = (float*) calloc((size_t)B*NH*T*T, sizeof(float));
    attention_pass1(preatt, inp, B, T, C, NH);
    FILE* f=fopen("course/ch06_build/preatt_ex1.bin","wb"); fwrite(preatt,4,(size_t)B*NH*T*T,f); fclose(f);
    free(inp); free(preatt); return 0;
}


In [ ]:
# Auto-grade Exercise 1
import numpy as np, torch, subprocess
torch.manual_seed(0); B, T, C, NH = 2, 4, 8, 2; hs = C // NH
q = torch.randn(B, T, C); k = torch.randn(B, T, C); v = torch.randn(B, T, C)
qkv = torch.cat([q, k, v], dim=-1)
qkv.numpy().astype(np.float32).tofile("course/ch06_build/inp.bin")
subprocess.run(["gcc","-O3","-Wall","-fopenmp","-o","course/ch06_build/exercise1","course/ch06_build/exercise1.c","-lm"], check=True)
subprocess.run(["./course/ch06_build/exercise1", str(B),str(T),str(C),str(NH)], check=True)
preatt_c = np.fromfile("course/ch06_build/preatt_ex1.bin", dtype=np.float32).reshape(B, NH, T, T)

# PT reference: only the t2 <= t positions matter
def heads(x): return x.view(B, T, NH, hs).transpose(1, 2)
qh, kh = heads(q), heads(k)
scores_pt = (qh @ kh.transpose(-2, -1)) * (hs ** -0.5)   # (B, NH, T, T)

# Compare only the lower triangle (where t2 <= t)
mask = torch.tril(torch.ones(T, T, dtype=torch.bool)).numpy()
err = np.max(np.abs((preatt_c - scores_pt.numpy())[:, :, mask]))
print(f"max diff (lower-triangle only): {err:.2e}")
print("PASS" if err < 1e-5 else "FAIL — check the dot product, the scale multiply, and the maxval update")


### Solution to Exercise 1

In [ ]:
%%writefile course/ch06_build/exercise1_sol.c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <omp.h>

void attention_pass1(float* preatt, float* inp, int B, int T, int C, int NH) {
    int C3 = C*3;
    int hs = C / NH;
    float scale = 1.0f / sqrtf((float)hs);

    #pragma omp parallel for collapse(3)
    for (int b = 0; b < B; b++)
        for (int t = 0; t < T; t++)
            for (int h = 0; h < NH; h++) {
                float* query_t    = inp    + b*T*C3 + t*C3 + h*hs;
                float* preatt_bth = preatt + b*NH*T*T + h*T*T + t*T;
                float maxval = -10000.0f;
                for (int t2 = 0; t2 <= t; t2++) {
                    float* key_t2 = inp + b*T*C3 + t2*C3 + h*hs + C;
                    float val = 0.0f;
                    for (int i = 0; i < hs; i++) val += query_t[i] * key_t2[i];
                    val *= scale;
                    if (val > maxval) maxval = val;
                    preatt_bth[t2] = val;
                }
                (void)maxval;
            }
}

static void* rd(const char* p, size_t n){FILE*f=fopen(p,"rb");void*b=malloc(n);size_t r=fread(b,1,n,f);(void)r;fclose(f);return b;}

int main(int argc, char** argv) {
    int B=atoi(argv[1]), T=atoi(argv[2]), C=atoi(argv[3]), NH=atoi(argv[4]);
    float* inp    = (float*) rd("course/ch06_build/inp.bin", (size_t)B*T*3*C*sizeof(float));
    float* preatt = (float*) calloc((size_t)B*NH*T*T, sizeof(float));
    attention_pass1(preatt, inp, B, T, C, NH);
    FILE* f=fopen("course/ch06_build/preatt_ex1.bin","wb"); fwrite(preatt,4,(size_t)B*NH*T*T,f); fclose(f);
    free(inp); free(preatt); return 0;
}


In [ ]:
!gcc -O3 -Wall -fopenmp -o course/ch06_build/exercise1_sol course/ch06_build/exercise1_sol.c -lm && ./course/ch06_build/exercise1_sol 2 4 8 2 && echo ran


## 12. TODO Exercise 2 — Pass 4 (Weighted Sum of V)

Given the post-softmax `att` weights, accumulate the values. **Don't forget to zero `out_bth` first.**


In [ ]:
%%writefile course/ch06_build/exercise2.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

void attention_pass4(float* out, float* att, float* inp, int B, int T, int C, int NH) {
    int C3 = C*3;
    int hs = C / NH;

    #pragma omp parallel for collapse(3)
    for (int b = 0; b < B; b++)
        for (int t = 0; t < T; t++)
            for (int h = 0; h < NH; h++) {
                float* att_bth = att + b*NH*T*T + h*T*T + t*T;
                float* out_bth = out + b*T*C    + t*C    + h*hs;

                // TODO 1: zero out_bth[0..hs)

                for (int t2 = 0; t2 <= t; t2++) {
                    float* value_t2 = inp + b*T*C3 + t2*C3 + h*hs + C*2;
                    float a = att_bth[t2];
                    // TODO 2: out_bth[i] += a * value_t2[i]  for i in [0, hs)
                }
            }
}

static void* rd(const char* p, size_t n){FILE*f=fopen(p,"rb");void*b=malloc(n);size_t r=fread(b,1,n,f);(void)r;fclose(f);return b;}

int main(int argc, char** argv) {
    int B=atoi(argv[1]), T=atoi(argv[2]), C=atoi(argv[3]), NH=atoi(argv[4]);
    float* inp = (float*) rd("course/ch06_build/inp.bin", (size_t)B*T*3*C*sizeof(float));
    float* att = (float*) rd("course/ch06_build/att.bin", (size_t)B*NH*T*T*sizeof(float));
    float* out = (float*) malloc((size_t)B*T*C*sizeof(float));
    attention_pass4(out, att, inp, B, T, C, NH);
    FILE* f = fopen("course/ch06_build/out_ex2.bin","wb"); fwrite(out,4,(size_t)B*T*C,f); fclose(f);
    free(inp); free(att); free(out); return 0;
}


In [ ]:
# Auto-grade Exercise 2 — uses the att.bin produced by the verified forward
import numpy as np, torch, torch.nn.functional as F, subprocess
torch.manual_seed(0); B, T, C, NH = 2, 4, 8, 2; hs = C // NH
q = torch.randn(B, T, C); k = torch.randn(B, T, C); v = torch.randn(B, T, C)
qkv = torch.cat([q, k, v], dim=-1)
qkv.numpy().astype(np.float32).tofile("course/ch06_build/inp.bin")

# Re-run the verified forward to get a fresh att.bin
subprocess.run(["./course/ch06_build/attention_forward", str(B),str(T),str(C),str(NH)], check=True)

# Build the same out_pt as in section 7 for reference
def heads(x): return x.view(B, T, NH, hs).transpose(1, 2)
qh, kh, vh = heads(q), heads(k), heads(v)
scores = (qh @ kh.transpose(-2, -1)) * (hs ** -0.5)
mask = torch.tril(torch.ones(T, T, dtype=torch.bool))
att_pt = F.softmax(scores.masked_fill(~mask, float('-inf')), dim=-1).masked_fill(~mask, 0.0)
out_pt = (att_pt @ vh).transpose(1, 2).contiguous().view(B, T, C)

subprocess.run(["gcc","-O3","-Wall","-fopenmp","-o","course/ch06_build/exercise2","course/ch06_build/exercise2.c"], check=True)
subprocess.run(["./course/ch06_build/exercise2", str(B),str(T),str(C),str(NH)], check=True)
out_ex = np.fromfile("course/ch06_build/out_ex2.bin", dtype=np.float32).reshape(B, T, C)
err = np.max(np.abs(out_ex - out_pt.numpy()))
print(f"max diff: {err:.2e}")
print("PASS" if err < 1e-5 else "FAIL — did you remember to zero out_bth first?")


### Solution to Exercise 2

In [ ]:
%%writefile course/ch06_build/exercise2_sol.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

void attention_pass4(float* out, float* att, float* inp, int B, int T, int C, int NH) {
    int C3 = C*3;
    int hs = C / NH;
    #pragma omp parallel for collapse(3)
    for (int b = 0; b < B; b++)
        for (int t = 0; t < T; t++)
            for (int h = 0; h < NH; h++) {
                float* att_bth = att + b*NH*T*T + h*T*T + t*T;
                float* out_bth = out + b*T*C    + t*C    + h*hs;
                for (int i = 0; i < hs; i++) out_bth[i] = 0.0f;
                for (int t2 = 0; t2 <= t; t2++) {
                    float* value_t2 = inp + b*T*C3 + t2*C3 + h*hs + C*2;
                    float a = att_bth[t2];
                    for (int i = 0; i < hs; i++) out_bth[i] += a * value_t2[i];
                }
            }
}

static void* rd(const char* p, size_t n){FILE*f=fopen(p,"rb");void*b=malloc(n);size_t r=fread(b,1,n,f);(void)r;fclose(f);return b;}

int main(int argc, char** argv) {
    int B=atoi(argv[1]), T=atoi(argv[2]), C=atoi(argv[3]), NH=atoi(argv[4]);
    float* inp = (float*) rd("course/ch06_build/inp.bin", (size_t)B*T*3*C*sizeof(float));
    float* att = (float*) rd("course/ch06_build/att.bin", (size_t)B*NH*T*T*sizeof(float));
    float* out = (float*) malloc((size_t)B*T*C*sizeof(float));
    attention_pass4(out, att, inp, B, T, C, NH);
    FILE* f = fopen("course/ch06_build/out_ex2.bin","wb"); fwrite(out,4,(size_t)B*T*C,f); fclose(f);
    free(inp); free(att); free(out); return 0;
}


In [ ]:
!gcc -O3 -Wall -fopenmp -o course/ch06_build/exercise2_sol course/ch06_build/exercise2_sol.c && ./course/ch06_build/exercise2_sol 2 4 8 2 && echo ran


## Recap

You now know:

- The QKV layout is `(B, T, 3*C)` — Q, K, V are *interleaved* in the last dim, and the C code addresses them with `+0`, `+C`, `+2*C` offsets.
- The forward is **four passes per `(b, t, h)`**: scaled Q·K, max-shift, exp+sum, normalize+mask, and value accumulation. Causality is implemented as `for (t2 = 0; t2 <= t)` — the upper triangle is simply *never visited*.
- Multi-head adds **one more `for h`** loop and one more `+ h*hs` offset. No new math.
- The backward has three stages — V accumulation, softmax Jacobian (the explicit `T × T` form), and the QK matmul — done in reverse order. Each stage scatters into the same packed `dinp` buffer at offsets `+0/+C/+2C`.
- The CPU backward is **not parallel** because Stages A and C have race conditions over `t2`. The CUDA version (Chapter 16) uses a different parallelism strategy that *is* race-free.

Attention is genuinely the hairiest layer in `train_gpt2.c`. After this, **everything else looks easy.**

### What's next

**Chapter 7 — Softmax + Cross-Entropy.** A short chapter on the loss head: numerically stable softmax (you've already seen the max-subtract trick), the negative-log-likelihood, and *why and when* the two get fused into one op (foreshadowing `fused_classifier.cuh` in Part III). We'll also get our first look at the padded vocab `Vp` vs real vocab `V`, which exists for nice GPU-friendly shapes.
